# Bank Loan Portfolio — Risk Analyst Report

Canonical Python companion to the Power BI dashboard and KPI contract.


## Summary

- Primary outcome metric: matured default rate among resolved loans.
- Review priority combines risk, funded exposure, and sample size.
- Results are descriptive monitoring signals, not approval rules or expected-loss estimates.


## Context & Methods

Metrics come from `src/risk_metrics.py` and `docs/metric_contract.md`. Only complete issue months are compared. Payment-date chronology is excluded until source semantics are verified.


In [1]:
from pathlib import Path
import sys
import pandas as pd

from IPython.display import display

root = Path.cwd().resolve()
for candidate in [root, *root.parents]:
    if (candidate / 'data' / 'financial_loan.csv').exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError('Cannot find data/financial_loan.csv')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.risk_metrics import (
    complete_month_comparison, load_loan_data, portfolio_kpis, segment_summary
)
loans = load_loan_data(PROJECT_ROOT / 'data' / 'financial_loan.csv')


## Data


In [2]:
print(f'Loan grain: {loans.id.nunique():,} unique IDs')
print(f'Issue-date range: {loans.issue_date.min():%Y-%m-%d} to {loans.issue_date.max():%Y-%m-%d}')
display(loans.head(10))


Loan grain: 38,576 unique IDs
Issue-date range: 2021-01-01 to 2021-12-12


,id,address_state,application_type,emp_length,emp_title,grade,home_ownership,issue_date,last_credit_pull_date,last_payment_date,...,installment,int_rate,loan_amount,total_acc,total_payment,is_resolved,is_charged_off,is_current,issue_month,dti_band
0,1077430,GA,INDIVIDUAL,< 1 year,Ryder,C,RENT,2021-02-11,13-09-2021,13-04-2021,...,59.83,0.1527,2500,4,1009,True,True,False,2021-02,≤15%
1,1072053,CA,INDIVIDUAL,9 years,MKC Accounting,E,RENT,2021-01-01,14-12-2021,15-01-2021,...,109.43,0.1864,3000,4,3939,True,False,False,2021-01,≤15%
2,1069243,CA,INDIVIDUAL,4 years,Chemat Technology Inc,C,RENT,2021-01-05,12-12-2021,09-01-2021,...,421.65,0.1596,12000,11,3522,True,True,False,2021-01,>20%
3,1041756,TX,INDIVIDUAL,< 1 year,barnes distribution,B,MORTGAGE,2021-02-25,12-12-2021,12-03-2021,...,97.06,0.1065,4500,9,4911,True,False,False,2021-02,≤15%
4,1068350,IL,INDIVIDUAL,10+ years,J&J Steel Inc,A,MORTGAGE,2021-01-01,14-12-2021,15-01-2021,...,106.53,0.0603,3500,28,3835,True,False,False,2021-01,≤15%
5,1062608,CA,INDIVIDUAL,3 years,Studio 94 Corp,C,RENT,2021-07-17,16-03-2021,12-08-2021,...,275.96,0.1465,8000,11,8637,True,False,False,2021-07,≤15%
6,1067441,TX,INDIVIDUAL,10+ years,American Airlines,C,MORTGAGE,2021-11-19,14-06-2021,13-12-2021,...,205.86,0.1427,6000,30,7218,True,False,False,2021-11,>20%
7,1066424,PA,INDIVIDUAL,10+ years,SCI Mahanoy,A,OWN,2021-06-11,14-07-2021,14-07-2021,...,172.10,0.0790,5500,23,6172,True,False,False,2021-06,≤15%
8,1065254,FL,INDIVIDUAL,10+ years,Tech Data Corp,A,MORTGAGE,2021-09-02,15-06-2021,12-10-2021,...,762.08,0.0890,24000,31,8650,True,True,False,2021-09,≤15%
9,1064589,MI,INDIVIDUAL,10+ years,teltow contracting,B,MORTGAGE,2021-02-09,16-03-2021,16-03-2021,...,93.21,0.1269,4125,21,5551,True,False,False,2021-02,15–20%


## Results

### 1. Portfolio health


In [3]:
portfolio = portfolio_kpis(loans)
display(portfolio.rename('value').to_frame())


,value
total_applications,3.857600e+04
funded_exposure,4.357571e+08
total_amount_collected,4.730709e+08
resolved_loans,3.747800e+04
charged_off_loans,5.333000e+03
current_loans,1.098000e+03
charge_off_share,1.382466e-01
matured_default_rate,1.422968e-01
current_loan_share,2.846329e-02
cash_collection_ratio,1.085630e+00


### 2. Grade risk and exposure


In [4]:
grade_risk = segment_summary(loans, 'grade').sort_values('grade')
display(grade_risk[['grade','loan_count','funded_exposure','resolved_loans','matured_default_rate','ci_low','ci_high','risk_exposure_proxy','review_priority']])


,grade,loan_count,funded_exposure,resolved_loans,matured_default_rate,ci_low,ci_high,risk_exposure_proxy,review_priority
0,A,9689,84252225,9654,0.057178,0.052721,0.061988,4.817405e+06,Monitor
1,B,11674,130703975,11347,0.118357,0.112542,0.124430,1.546977e+07,Monitor
2,C,7904,87456450,7647,0.165555,0.157393,0.174053,1.447886e+07,Medium
3,D,5182,63920800,4966,0.215868,0.204647,0.227528,1.379845e+07,High
4,E,2786,44165100,2611,0.264650,0.248083,0.281907,1.168827e+07,High
5,F,1028,18910450,957,0.324974,0.296051,0.355297,6.145402e+06,High
6,G,313,6348075,296,0.331081,0.279935,0.386556,2.101728e+06,Watch: small segment


### 3. State concentration


In [5]:
state_risk = segment_summary(loans, 'address_state').sort_values('funded_exposure', ascending=False)
display(state_risk.head(10)[['address_state','funded_exposure','funded_exposure_share','resolved_loans','matured_default_rate','default_rate_vs_portfolio_pp','review_priority']])


,address_state,funded_exposure,funded_exposure_share,resolved_loans,matured_default_rate,default_rate_vs_portfolio_pp,review_priority
4,CA,78484125,0.180110,6751,0.156569,1.427258,Medium
33,NY,42077050,0.096561,3591,0.130326,-1.197100,Monitor
42,TX,31236650,0.071684,2597,0.115903,-2.639385,Monitor
9,FL,30046125,0.068952,2691,0.178001,3.570393,High
30,NJ,21657475,0.049701,1765,0.154674,1.237741,Medium
14,IL,17124225,0.039298,1440,0.133333,-0.896348,Monitor
44,VA,15982650,0.036678,1337,0.127150,-1.514648,Monitor
37,PA,15826525,0.036320,1436,0.117688,-2.460879,Monitor
10,GA,15480325,0.035525,1317,0.157175,1.487858,Medium
19,MA,15051000,0.034540,1268,0.118297,-2.400028,Monitor


### 4. Purpose × term review matrix


In [6]:
purpose_term_risk = segment_summary(loans, ['purpose','term']).sort_values('risk_exposure_proxy', ascending=False)
display(purpose_term_risk.head(12)[['purpose','term','funded_exposure','resolved_loans','matured_default_rate','default_rate_vs_portfolio_pp','risk_exposure_proxy','review_priority']])


,purpose,term,funded_exposure,resolved_loans,matured_default_rate,default_rate_vs_portfolio_pp,risk_exposure_proxy,review_priority
1,Debt consolidation,60 months,92775900,4824,0.261194,11.889722,2.423251e+07,High
0,Debt consolidation,36 months,139683775,12823,0.108477,-3.381986,1.515247e+07,Monitor
23,small business,60 months,10181175,500,0.396000,25.370319,4.031745e+06,High
5,credit card,60 months,17122450,915,0.231694,8.939717,3.967169e+06,High
4,credit card,36 months,41762725,3982,0.074335,-6.796231,3.104412e+06,Monitor
22,small business,36 months,13941925,1203,0.213633,7.133577,2.978449e+06,High
19,other,60 months,9861000,695,0.293525,15.122837,2.894452e+06,High
18,other,36 months,21294750,3009,0.127285,-1.501200,2.710498e+06,Monitor
9,home improvement,60 months,14279700,773,0.175938,3.364109,2.512340e+06,High
8,home improvement,36 months,19071075,2009,0.095072,-4.722464,1.813129e+06,Monitor


### 5. Latest complete issue-month comparison


In [7]:
display(complete_month_comparison(loans))


,issue_month,total_applications,funded_exposure,total_amount_collected,average_interest_rate,average_dti,total_applications_mom,funded_exposure_mom,total_amount_collected_mom,average_interest_rate_mom,average_dti_mom
0,2021-10,3796,44893800.0,49399567.0,0.120241,0.134144,NaN,NaN,NaN,NaN,NaN
1,2021-11,4035,47754825.0,50132030.0,0.119417,0.133027,0.062961,0.063729,0.014827,-0.006853,-0.008323


## Takeaways

1. **Headline risk:** Prioritize a grade-controlled policy and pricing backtest for 60-month debt-consolidation loans; the segment combines material exposure, a 26.1% matured default rate, and higher rates than the 36-month comparison within Grades A–F.
2. **Opportunity hypothesis:** Investigate why Grade A has lower funded exposure than Grades B and C despite its lower observed default rate; do not treat this as an automatic expansion rule.
3. **State monitoring:** Treat Florida as the stronger risk hypothesis after accounting for grade mix, while treating California primarily as a concentration watch item.
4. **Supporting guardrail:** Retain DTI above 20% as a monitoring variable, not as evidence for a hard underwriting cutoff.
5. Keep payment timing, expected loss, profitability, and causal policy decisions out of scope until governed source data is available.
